# 03 — LoRA Adapters

Objetivo: entender o que LoRA faz de diferente do full fine-tuning.

**Conceito central:** LoRA congela todos os pesos do modelo base e injeta matrizes treináveis de rank baixo nas camadas de atenção.

```
W_adapted = W_frozen + (A @ B) * (alpha / r)
```

Onde:
- `A ∈ R^(d×r)` e `B ∈ R^(r×k)` são as matrizes LoRA
- `r` (rank) é muito menor que `d` — tipicamente 4, 8, 16
- Resultado: ~0.3% dos parâmetros do modelo são treináveis

Ao final deste notebook você vai comparar full fine-tuning vs LoRA em accuracy, F1, tempo de treino e tamanho do artefato.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../src').resolve()))

import torch
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.dpi'] = 120

## 1. Inspecionar parâmetros treináveis antes de treinar

Essa célula mostra exatamente o que LoRA faz com o modelo.

In [ ]:
from model import load_model
from peft import LoraConfig, get_peft_model, TaskType

base_model = load_model()

# Contar parâmetros ANTES do LoRA
total_params = sum(p.numel() for p in base_model.parameters())
print(f'Base model (CodeBERT) — total params: {total_params:,}')

# Aplicar LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['query', 'value'],
    bias='none',
)
lora_model = get_peft_model(base_model, lora_config)

print()
lora_model.print_trainable_parameters()

trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
frozen = total_params - trainable
print(f'\nCongelados: {frozen:,}')
print(f'Treináveis: {trainable:,}')
print(f'Razão: 1:{frozen//trainable} (congelados:treináveis)')

## 2. Visualizar quais camadas têm parâmetros treináveis

In [ ]:
layer_data = []
for name, param in lora_model.named_parameters():
    layer_data.append({
        'layer': name,
        'params': param.numel(),
        'trainable': param.requires_grad,
    })

df = pd.DataFrame(layer_data)
trainable_layers = df[df['trainable']]
print(f'Camadas com parâmetros treináveis ({len(trainable_layers)}):')
print(trainable_layers[['layer', 'params']].to_string(index=False))

## 3. Treinar com LoRA

In [ ]:
from lora_train import train_lora

train_lora(
    r=16,
    lora_alpha=32,
    num_epochs=5,
    batch_size=16,
    learning_rate=2e-5,
)

## 4. Comparação: Full Fine-tuning vs LoRA

In [ ]:
import mlflow
import os

mlflow.set_tracking_uri("sqlite:///" + str(Path('..').resolve() / 'mlflow.db'))
client = mlflow.tracking.MlflowClient()

def get_best_metrics(experiment_name):
    exp = client.get_experiment_by_name(experiment_name)
    if not exp:
        return None
    runs = client.search_runs(experiment_ids=[exp.experiment_id], order_by=['start_time DESC'])
    if not runs:
        return None
    return runs[0].data.metrics

full_metrics = get_best_metrics('code-review-classifier')
lora_metrics = get_best_metrics('code-review-classifier-lora')

if full_metrics and lora_metrics:
    comparison = pd.DataFrame({
        'Full Fine-tuning': full_metrics,
        'LoRA': lora_metrics,
    })
    print(comparison)

In [ ]:
# Tamanho dos artefatos
def dir_size_mb(path):
    p = Path(path)
    if not p.exists():
        return None
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1024 / 1024

full_size = dir_size_mb('../models/full_finetuned')
lora_size = dir_size_mb('../models/lora_adapter')

print('Tamanho dos artefatos:')
print(f'  Full fine-tuning: {full_size:.1f} MB' if full_size else '  Full fine-tuning: não encontrado')
print(f'  LoRA adapter:     {lora_size:.1f} MB' if lora_size else '  LoRA adapter: não encontrado')
if full_size and lora_size:
    print(f'  Redução: {full_size/lora_size:.0f}x menor com LoRA')

## 5. Tabela de comparação final

In [ ]:
from evaluate import evaluate
from sklearn.metrics import f1_score
import torch
from peft import PeftModel
from model import load_finetuned, LABELS, LABEL2ID
from torch.utils.data import DataLoader
from train import ReviewDataset

# Avaliar full fine-tuning
print('=== Full Fine-tuning ===')
full_results = evaluate(checkpoint_path=Path('../models/full_finetuned'))

# Avaliar LoRA — carregar base fine-tunado + adapter
print('\n=== LoRA Adapter ===')
base_model, tokenizer = load_finetuned('../models/full_finetuned')
lora_model = PeftModel.from_pretrained(base_model, '../models/lora_adapter')
lora_model.eval()

test_ds = ReviewDataset(Path('../data/splits/test.jsonl'), tokenizer)
test_loader = DataLoader(test_ds, batch_size=32)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
lora_model.to(device)

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        outputs = lora_model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
        )
        preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch['labels'].tolist())

lora_f1 = f1_score(all_labels, all_preds, average='macro')
print(f'F1 Macro LoRA: {lora_f1:.4f}')

full_size = dir_size_mb('../models/full_finetuned')
lora_size = dir_size_mb('../models/lora_adapter')

# Tabela final
print('\n' + '='*58)
print(f"{'Métrica':25} | {'Full FT':14} | {'LoRA':12}")
print('-'*58)
print(f"{'F1 Macro':25} | {full_results['f1_macro']:.4f}         | {lora_f1:.4f}")
print(f"{'Params treináveis':25} | {'125M (100%)':14} | {'1.18M (0.94%)':12}")
if full_size and lora_size:
    print(f"{'Tamanho artefato':25} | {full_size:.1f} MB         | {lora_size:.1f} MB")
    print(f"{'Redução':25} | {'—':14} | {full_size/lora_size:.0f}x menor")
print('='*58)

## 6. O que aprendemos

1. **LoRA funciona com ~0.3% dos parâmetros** — os adapters A e B capturam a adaptação de domínio sem modificar os pesos pré-treinados
2. **O modelo base permanece intacto** — você pode carregar diferentes adapters sobre o mesmo base model (um por cliente, um por domínio)
3. **Trade-off real**: LoRA perde alguns pontos de F1 vs full fine-tuning, mas o ganho em tamanho e custo de treino é enorme
4. **Isso é como GPT fine-tuned, Llama adapters, etc funcionam** — mesma mecânica, escala maior